# שיעור 07 - תבנית עיצוב תכנון

פנקס רשימות זה מדגים את **תבנית העיצוב תכנון** לסוכני בינה מלאכותית באמצעות מסגרת העבודה של Microsoft Agent.
תלמד כיצד לפרק בקשות נסיעה מורכבות למשימות משנה מובנות, להקצות אותן לסוכנים מומחים,
ולבצע את התכנית המתבקשת — הכל באמצעות פלט מובנה המופעל על ידי מודלים של Pydantic.


## הגדרה


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## פירוק משימה

פירוק משימה הוא ליבת תבנית העיצוב לתכנון. במקום לבקש מסוכן יחיד
לטפל בבקשה מורכבת מתחילתה ועד סופה, אנו מפרקים את הבעיה ל**משימות משנה** קטנות ומוגדרות היטב.
כל משימת משנה מוקצת לסוכן מומחה (למשל, טיסות, בתי מלון, פעילויות) עם
עדיפויות ברורות וסדר תלות.

גישה זו מספקת מספר יתרונות:
- **בהירות**: לכל משימת משנה יש אחריות יחידה.
- **מקביליות**: משימות משנה בלתי תלויות יכולות לרוץ במקביל.
- **אמינות**: תקלות מבודדות למשימות משנה פרטניות.
- **מעקב תקציבי**: העלויות מוערכות לפי משימת משנה ומצטברות.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## יצירת סוכן תכנון עם פלט מובנה

סוכן התכנון פועל כ**רכז קבלה**. בהתחשב בבקשת נסיעה ברמה גבוהה, הוא
מייצר `TravelPlan` מובנה — מפצל את הבקשה לתת-משימות, מגדיר עדיפויות,
ומזהה תלותיות כך שסוכן שירות או שכבת ביצוע יוכלו לבצע את העבודה.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## ביצוע תכנית עם כלים ספציאליסטיים

ברגע שסוכן הקבלה יצר תכנית מובנית, **הסוכן הקונסיירז'** מבצע אותה.
כל כלי ספציאליסט מטפל בקטגוריה אחת של תת-משימות (טיסות, מלונות, פעילויות). הקונסיירז'
עובר על תת-המשימות בתכנית לפי סדר תלות ושולח כל אחת לכלי המתאים.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## סיכום

בשיעור זה למדת את **תבנית התכנון** לסוכני בינה מלאכותית:

1. **פירוק משימה** — סוכן תכנון שמכיר את הנושא מפצל בקשה מורכבת לנסיעה למטלות משנה
   מסודרות באמצעות מודלי Pydantic, ומקצה כל אחת לסוכן מומחה עם עדיפויות
   ותלויות.
2. **פלט מובנה** — על ידי העברת `response_format` הסוכן מחזיר אובייקט `TravelPlan`
   מאומת במקום טקסט חופשי, מה שהופך את העיבוד הבא למהימן.
3. **ביצוע התוכנית** — סוכן קונסיורג' עובר על מטלות המשנה באמצעות כלים מומחים
   (`book_flight`, `reserve_hotel`, `book_activity`) כדי לבצע את התוכנית ולדווח על התוצאות.

תבנית זו מפרידה בין *מה לעשות* (תכנון) ל*איך לעשות* (ביצוע), מה שהופך את הסוכנים
ליותר מודולריים, ניתנים לבדיקה, וקלים יותר להרחבה.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
